In [2]:
from elasticsearch import Elasticsearch

In [3]:
es= Elasticsearch(
    "https://localhost:9200",
    basic_auth=("elastic","nS=4h9oZdXlTLSKj=9NT"),
    ca_certs="/home/sonu/Desktop/semantic_search/certs/http_ca.crt"
)
es.ping()

True

## Prepare data

In [4]:
import pandas as pd
df=pd.read_csv("/home/sonu/Desktop/semantic_search/myntra_products_catalog.csv").loc[:499]

In [5]:
df.head()

,ProductID,ProductName,ProductBrand,Gender,Price (INR),NumImages,Description,PrimaryColor
0,10017413,DKNY Unisex Black & Grey Printed Medium Trolle...,DKNY,Unisex,11745,7,"Black and grey printed medium trolley bag, sec...",Black
1,10016283,EthnoVogue Women Beige & Grey Made to Measure ...,EthnoVogue,Women,5810,7,Beige & Grey made to measure kurta with churid...,Beige
2,10009781,SPYKAR Women Pink Alexa Super Skinny Fit High-...,SPYKAR,Women,899,7,Pink coloured wash 5-pocket high-rise cropped ...,Pink
3,10015921,Raymond Men Blue Self-Design Single-Breasted B...,Raymond,Men,5599,5,Blue self-design bandhgala suitBlue self-desig...,Blue
4,10017833,Parx Men Brown & Off-White Slim Fit Printed Ca...,Parx,Men,759,5,"Brown and off-white printed casual shirt, has ...",White


In [6]:
df.isnull().sum()

ProductID        0
ProductName      0
ProductBrand     0
Gender           0
Price (INR)      0
NumImages        0
Description      0
PrimaryColor    32
dtype: int64

In [8]:
df.fillna("None",inplace=True)

,ProductID,ProductName,ProductBrand,Gender,Price (INR),NumImages,Description,PrimaryColor
0,10017413,DKNY Unisex Black & Grey Printed Medium Trolle...,DKNY,Unisex,11745,7,"Black and grey printed medium trolley bag, sec...",Black
1,10016283,EthnoVogue Women Beige & Grey Made to Measure ...,EthnoVogue,Women,5810,7,Beige & Grey made to measure kurta with churid...,Beige
2,10009781,SPYKAR Women Pink Alexa Super Skinny Fit High-...,SPYKAR,Women,899,7,Pink coloured wash 5-pocket high-rise cropped ...,Pink
3,10015921,Raymond Men Blue Self-Design Single-Breasted B...,Raymond,Men,5599,5,Blue self-design bandhgala suitBlue self-desig...,Blue
4,10017833,Parx Men Brown & Off-White Slim Fit Printed Ca...,Parx,Men,759,5,"Brown and off-white printed casual shirt, has ...",White
...,...,...,...,...,...,...,...,...
495,10018075,Puma Men Blue Sneakers,Puma,Men,1749,5,"A pair of round-toe blue sneakers, has regular...",Blue
496,10009687,SPYKAR Women Blue & White Striped Adora Skinny...,SPYKAR,Women,1034,7,Blue and White striped 5-pocket mid-rise jeans...,Blue
497,10017425,DKNY Unisex Black & Grey Printed Large Trolley...,DKNY,Unisex,13275,7,"Black and grey printed large trolley bag, secu...",Black
498,10017615,Parx Men Pink Slim Fit Solid Casual Shirt,Parx,Men,612,5,"Pink solid casual shirt, has a spread collar, ...",Pink


## Convert to vectors

In [11]:
from sentence_transformers import SentenceTransformer

# Load https://huggingface.co/sentence-transformers/all-mpnet-base-v2
model = SentenceTransformer("all-mpnet-base-v2")

/home/sonu/Desktop/semantic_search/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 462.95it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
df["DescriptionVector"]=df["Description"].apply(lambda x: model.encode(x))

In [13]:
df.head()

,ProductID,ProductName,ProductBrand,Gender,Price (INR),NumImages,Description,PrimaryColor,DescriptionVector
0,10017413,DKNY Unisex Black & Grey Printed Medium Trolle...,DKNY,Unisex,11745,7,"Black and grey printed medium trolley bag, sec...",Black,"[0.027645921, -0.0026341653, -0.0035884266, 0...."
1,10016283,EthnoVogue Women Beige & Grey Made to Measure ...,EthnoVogue,Women,5810,7,Beige & Grey made to measure kurta with churid...,Beige,"[-0.024660701, -0.028755344, -0.020332504, 0.0..."
2,10009781,SPYKAR Women Pink Alexa Super Skinny Fit High-...,SPYKAR,Women,899,7,Pink coloured wash 5-pocket high-rise cropped ...,Pink,"[-0.046943232, 0.08182793, 0.048335142, -0.000..."
3,10015921,Raymond Men Blue Self-Design Single-Breasted B...,Raymond,Men,5599,5,Blue self-design bandhgala suitBlue self-desig...,Blue,"[-0.0150987515, -0.010285409, 0.00948729, -0.0..."
4,10017833,Parx Men Brown & Off-White Slim Fit Printed Ca...,Parx,Men,759,5,"Brown and off-white printed casual shirt, has ...",White,"[-0.017746592, 0.006209664, 0.021813968, 0.026..."
